# Cross-Axis Capping: A Primer

This notebook walks you through **every concept** used in the cross-capping experiment, starting from the absolute basics. By the end you'll understand:

- How to load a model and dataset from HuggingFace
- How tokenization and chat templates work
- What transformer layers and hidden states are
- How to use **forward hooks** to read and modify activations
- What projections and dot products mean geometrically
- How **capping** works as a safety intervention
- How **cross-axis capping** splits detection from correction
- How to compute thresholds and build axes from data
- How to parallelise across multiple GPUs

**We use GPT-2 (124M parameters)** for the runnable examples so you can follow along on any machine. The real experiment uses Qwen3-32B (32B parameters), but all the concepts are identical -- just bigger tensors.

> **Prerequisites:** Basic Python, some familiarity with PyTorch tensors. No prior knowledge of transformers or mechanistic interpretability needed.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers datasets huggingface_hub scikit-learn numpy

---
## Part 1: Loading a Model from HuggingFace

HuggingFace hosts thousands of pre-trained models. The `transformers` library lets you load any of them with two lines of code:

- **`AutoModelForCausalLM`** loads the model itself (the neural network with all its weights)
- **`AutoTokenizer`** loads the tokenizer (converts text to numbers and back)

"Causal LM" means the model generates text left-to-right, one token at a time -- which is what GPT-2, LLaMA, Qwen, etc. all do.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load GPT-2 (124M params, ~500MB -- downloads automatically on first run)
model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Put the model in evaluation mode (disables dropout, etc.)
model.eval()

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

### Scaling up: dtype and device_map

GPT-2 is small enough to run on CPU. Larger models need GPUs and reduced precision:

```python
# In the real experiment we load Qwen3-32B like this:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-32B",
    dtype=torch.bfloat16,          # half-precision: halves memory usage
    device_map="auto",             # automatically spread layers across available GPUs
    attn_implementation="flash_attention_2",  # faster, more memory-efficient attention
)
```

- **`dtype=torch.bfloat16`**: Stores each weight in 16 bits instead of 32. A 32B-parameter model needs ~64GB in float32 but ~32GB in bfloat16. 
- **`device_map="auto"`**: If you have multiple GPUs, this places different layers on different GPUs automatically.
- **`flash_attention_2`**: A faster implementation of the attention mechanism that uses less memory.

---
## Part 2: Loading Datasets from HuggingFace

HuggingFace also hosts datasets. The `datasets` library downloads and caches them for you.

In [ ]:
from datasets import load_dataset

# Load a small dataset as an example
# This is the "harmful behaviors" dataset from JailbreakBench
ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")

# Datasets are dict-like: keys are split names ("train", "test", etc.)
print("Available splits:", list(ds.keys()))

# Grab the first split and look at a row
split = list(ds.keys())[0]
print(f"\nNumber of rows: {len(ds[split])}")
print(f"Columns: {ds[split].column_names}")
print(f"\nExample goal: {ds[split][0]['Goal'][:100]}...")

In the real experiment we use three datasets:

| Dataset | What it contains | How we use it |
|---|---|---|
| **JBB-Behaviors** | Bare harmful goals (no jailbreak wrapper) | Model refuses these -- we collect "refusing" activations |
| **WildJailbreak (train)** | Jailbreak prompts that trick the model | Model complies -- we collect "compliant" activations |
| **WildJailbreak (eval)** | Held-out jailbreak prompts | The actual test set we evaluate capping on |

The key idea: by comparing activations from prompts the model refuses vs. prompts it complies with, we can find the direction in activation space that separates the two.

---
## Part 3: Tokenization and Chat Templates

Models don't read text -- they read **tokens** (integers). A tokenizer converts between the two.

### Basic tokenization

In [ ]:
# Tokenize a sentence
text = "The cat sat on the mat"
tokens = tokenizer(text, return_tensors="pt")

print("Original text:", text)
print("Token IDs:    ", tokens["input_ids"][0].tolist())
print("Decoded back: ", tokenizer.decode(tokens["input_ids"][0]))

# You can see how individual words map to token IDs
for token_id in tokens["input_ids"][0]:
    print(f"  {token_id.item():5d} -> '{tokenizer.decode(token_id)}'")

# Note: some words get split into multiple tokens (subword tokenization)
text2 = "Unbelievably extraordinary"
tokens2 = tokenizer(text2, return_tensors="pt")
for token_id in tokens2["input_ids"][0]:
    print(f"  {token_id.item():5d} -> '{tokenizer.decode(token_id)}'")

### Chat templates

Chat models (like Qwen3, LLaMA-Chat, etc.) expect input in a specific format with role markers. The tokenizer's `apply_chat_template` handles this:

```python
# For Qwen3, a user message gets wrapped like:
# <|im_start|>user
# What is the capital of France?<|im_end|>
# <|im_start|>assistant

conversation = [{"role": "user", "content": "What is the capital of France?"}]
text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
```

The `add_generation_prompt=True` appends the assistant's turn marker so the model knows to start generating a response.

GPT-2 is not a chat model, so it doesn't have a chat template. But the concept is the same for all chat models in the experiment.

---
## Part 4: Generating Text

A language model generates text one token at a time. At each step it:
1. Looks at all the tokens so far
2. Computes a probability distribution over the entire vocabulary
3. Picks the next token (greedily = always the most probable, or sampled = random weighted by probability)
4. Appends it and repeats

In [ ]:
# Generate some text with GPT-2
prompt = "The meaning of life is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

print(f"Prompt: '{prompt}'")
print(f"Prompt token IDs: {input_ids[0].tolist()}")
print(f"Prompt length: {input_ids.shape[1]} tokens\n")

# Generate 50 new tokens (greedy decoding -- deterministic, always picks the top token)
with torch.inference_mode():  # disables gradient tracking for speed
    output = model.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=False,       # greedy -- no randomness
    )

# Decode only the NEW tokens (everything after the prompt)
generated_text = tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Generated: '{generated_text}'")

---
## Part 5: What's Inside a Transformer?

A transformer model is a stack of **layers** (also called "blocks"). Each layer transforms a sequence of hidden state vectors. The data flows like this:

```
Input tokens  →  Embedding  →  Layer 0  →  Layer 1  →  ...  →  Layer N  →  Output logits
                                  ↑            ↑                    ↑
                            hidden states  hidden states       hidden states
```

At each layer, every token position has a **hidden state vector** -- a list of numbers (e.g. 768 numbers for GPT-2, 5120 for Qwen3-32B) that encodes everything the model "knows" about the text at that point.

The key insight for our experiment: **these hidden state vectors carry directional information about the model's intent**. If the model is about to refuse a request, its hidden states point in a different direction than if it's about to comply.

In [ ]:
# Let's look at GPT-2's internal structure

# GPT-2 stores its transformer layers in model.transformer.h
layers = model.transformer.h
print(f"Number of layers: {len(layers)}")
print(f"Hidden dimension: {model.config.n_embd}")  # 768 for GPT-2

# Each layer is a TransformerBlock with attention + feed-forward
print(f"\nFirst layer structure:")
print(layers[0])

# In the real experiment, different architectures store layers in different places:
# - GPT-2:          model.transformer.h
# - Qwen/LLaMA:     model.model.layers
# - Gemma 3:        model.language_model.layers
# crosscap_experiment.py handles this automatically with _get_layers()

---
## Part 6: Forward Hooks -- Reading Hidden States

This is one of the most important concepts in the experiment. A **forward hook** is a function that PyTorch calls every time data passes through a layer. You can use it to:

1. **Read** the hidden states (for monitoring/analysis)
2. **Modify** the hidden states (for intervention)

Think of it like putting a sensor on a pipe -- you can measure what flows through, or you can inject something into the flow.

### Reading activations with hooks

In [ ]:
# Let's attach a hook to layer 5 and see what the hidden state looks like

captured_activations = {}

def read_hook(module, input, output):
    """This function is called every time data passes through the layer.
    
    Args:
        module: the layer itself
        input:  what went INTO the layer
        output: what came OUT of the layer (this is what we want)
    """
    # output can be a tensor or a tuple; grab the hidden states
    hidden_states = output[0] if isinstance(output, tuple) else output
    
    # hidden_states shape: (batch_size, sequence_length, hidden_dim)
    # We care about the LAST token position (that's where the model
    # decides what to generate next)
    last_token_state = hidden_states[0, -1, :]  # [hidden_dim]
    
    captured_activations["layer_5"] = last_token_state.detach().clone()

# Register the hook on layer 5
handle = layers[5].register_forward_hook(read_hook)

# Run a forward pass -- the hook fires automatically
input_ids = tokenizer("Hello world", return_tensors="pt")["input_ids"]
with torch.inference_mode():
    model(input_ids)

# Remove the hook (important! otherwise it fires on every future forward pass)
handle.remove()

# Look at what we captured
h = captured_activations["layer_5"]
print(f"Hidden state shape: {h.shape}")        # [768] for GPT-2
print(f"Hidden state norm:  {h.norm():.2f}")    # how "big" the vector is
print(f"First 10 values:    {h[:10].tolist()}")  # just the first 10 numbers

### Capturing activations at EVERY layer

In the experiment we often want to see what happens at all layers at once. Here's how:

In [ ]:
# Capture activations at every layer in one forward pass

all_activations = {}
handles = []

for layer_idx in range(len(layers)):
    # We need a factory function because of Python's closure rules:
    # without make_hook, all hooks would capture the same layer_idx value
    def make_hook(idx):
        def hook_fn(module, input, output):
            h = output[0] if isinstance(output, tuple) else output
            all_activations[idx] = h[0, -1, :].detach().clone()
        return hook_fn
    
    handle = layers[layer_idx].register_forward_hook(make_hook(layer_idx))
    handles.append(handle)

# Run forward pass (all hooks fire)
input_ids = tokenizer("The quick brown fox", return_tensors="pt")["input_ids"]
with torch.inference_mode():
    model(input_ids)

# Clean up all hooks
for h in handles:
    h.remove()

# Now we have the hidden state at every layer
print(f"Captured {len(all_activations)} layers")
for idx in [0, 5, 11]:  # first, middle, last
    print(f"  Layer {idx:2d}: norm = {all_activations[idx].norm():.2f}")

# This is exactly what SteeringExperiment.get_baseline_trajectory() does
# in crosscap_experiment.py

---
## Part 7: Projections and Dot Products

This is the mathematical heart of the experiment. If you understand this section, everything else follows.

### The geometric intuition

A hidden state is a vector in a high-dimensional space (768 dimensions for GPT-2). An **axis** is a direction in that space. The **projection** (dot product) tells you how much the hidden state points in that direction.

Think of it like a compass:
- The axis is "north"
- The projection tells you how far north you are
- A high projection means the vector points strongly in that direction
- A low (or negative) projection means it points away

In our experiment:
- The **assistant axis** points in the direction of "helpful assistant behavior"
- When the model is about to comply with a jailbreak, its hidden state has a **low** projection on this axis
- When it's behaving normally, the projection is **high**

In [ ]:
import numpy as np

# Simple 2D example to build intuition
# Imagine we're in 2D space instead of 768D

# Define an axis (a unit vector pointing "north-east")
axis = torch.tensor([0.707, 0.707])  # normalised to length 1
print(f"Axis: {axis.tolist()}, norm: {axis.norm():.3f}")

# Vector A: points strongly in the axis direction
vector_a = torch.tensor([3.0, 4.0])
proj_a = vector_a @ axis  # dot product = projection
print(f"\nVector A: {vector_a.tolist()}")
print(f"Projection onto axis: {proj_a:.2f}")
print("  → High projection: A points in a similar direction to the axis")

# Vector B: points away from the axis direction
vector_b = torch.tensor([-2.0, 1.0])
proj_b = vector_b @ axis
print(f"\nVector B: {vector_b.tolist()}")
print(f"Projection onto axis: {proj_b:.2f}")
print("  → Low projection: B points mostly away from the axis")

# Vector C: perpendicular to the axis
vector_c = torch.tensor([3.0, -3.0])
proj_c = vector_c @ axis
print(f"\nVector C: {vector_c.tolist()}")
print(f"Projection onto axis: {proj_c:.2f}")
print("  → Near-zero projection: C is perpendicular to the axis")

### Projecting real hidden states onto a random axis

Let's see how this works with actual model activations. We'll create a random axis (since we don't have a real assistant axis for GPT-2) and project hidden states onto it.

In [ ]:
# Create a random unit axis in 768D (just for demonstration)
torch.manual_seed(42)
random_axis = torch.randn(model.config.n_embd)
random_axis = random_axis / random_axis.norm()  # normalise to unit length

# Project our previously captured hidden states onto this axis
print("Projection of each layer's hidden state onto a random axis:\n")
for layer_idx in range(len(layers)):
    h = all_activations[layer_idx].float()
    projection = (h @ random_axis).item()
    bar = "█" * int(abs(projection) * 2)
    sign = "+" if projection >= 0 else "-"
    print(f"  Layer {layer_idx:2d}: {projection:+7.2f}  {sign}{bar}")

# In the real experiment, the axis is NOT random -- it's the "assistant axis",
# a meaningful direction that was computed from model behavior.
# The projection values tell us whether the model is in "assistant mode"
# or "jailbreak-compliance mode".

---
## Part 8: Modifying Activations with Hooks

So far we've only *read* hidden states. Now let's *change* them. This is the core mechanism behind capping.

A hook can return a modified output, and PyTorch will use that modified version for the rest of the computation. This lets us "steer" the model's behavior by nudging its internal representations.

In [ ]:
# First, generate WITHOUT any hook (baseline)
prompt = "I think the best way to"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

with torch.inference_mode():
    baseline_output = model.generate(input_ids, max_new_tokens=30, do_sample=False)
baseline_text = tokenizer.decode(baseline_output[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Baseline: '{baseline_text}'\n")

# Now let's add a hook that MODIFIES the hidden state at layer 8
# We'll add a random perturbation to see how it changes the output
torch.manual_seed(123)
perturbation = torch.randn(model.config.n_embd) * 20  # large perturbation

def modify_hook(module, input, output):
    """Hook that adds a perturbation to the last token's hidden state."""
    if torch.is_tensor(output):
        h = output
    else:
        h = output[0]
    
    # Add the perturbation to the last token position
    h[0, -1, :] += perturbation
    
    # Return the modified output
    if torch.is_tensor(output):
        return h
    return (h, *output[1:])

# Generate WITH the hook
handle = layers[8].register_forward_hook(modify_hook)
with torch.inference_mode():
    modified_output = model.generate(input_ids, max_new_tokens=30, do_sample=False)
handle.remove()

modified_text = tokenizer.decode(modified_output[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Modified: '{modified_text}'")
print("\nNotice how changing the hidden state changes the generated text!")
print("This is the same mechanism capping uses -- but in a controlled, purposeful way.")

---
## Part 9: Capping -- The Safety Intervention

Now we can put the pieces together. **Capping** is a targeted modification: instead of adding a random perturbation, we nudge the hidden state along a specific axis by a precise amount.

### The formula

```
h ← h − v × min(⟨h, v⟩ − τ, 0)
```

In plain English:
1. **Measure**: compute the dot product `⟨h, v⟩` (how far does h project onto axis v?)
2. **Compare**: is the projection below the threshold τ?
3. **If yes, correct**: add just enough of v to bring the projection up to τ
4. **If no**: do nothing (the model is behaving normally)

The correction `min(⟨h, v⟩ − τ, 0)` is zero when the projection is already above τ, and negative when it's below -- so the hook only fires when needed.

In [ ]:
# Let's implement capping step by step in 2D to see how it works

# Our axis (unit vector)
v = torch.tensor([0.707, 0.707])

# A threshold -- projections below this get corrected
tau = 2.0

# Case 1: Hidden state that's ABOVE the threshold (no correction needed)
h1 = torch.tensor([3.0, 4.0])
proj1 = (h1 @ v).item()
correction1 = min(proj1 - tau, 0)   # min(4.95 - 2.0, 0) = min(2.95, 0) = 0
h1_capped = h1 - v * correction1

print("Case 1: Above threshold")
print(f"  h = {h1.tolist()}, projection = {proj1:.2f}")
print(f"  correction = min({proj1:.2f} - {tau}, 0) = {correction1:.2f}")
print(f"  h_capped = {h1_capped.tolist()} (unchanged!)\n")

# Case 2: Hidden state that's BELOW the threshold (correction needed)
h2 = torch.tensor([0.5, 0.5])
proj2 = (h2 @ v).item()
correction2 = min(proj2 - tau, 0)   # min(0.71 - 2.0, 0) = min(-1.29, 0) = -1.29
h2_capped = h2 - v * correction2   # subtracting a negative = adding

print("Case 2: Below threshold")
print(f"  h = {h2.tolist()}, projection = {proj2:.2f}")
print(f"  correction = min({proj2:.2f} - {tau}, 0) = {correction2:.2f}")
print(f"  h_capped = {[round(x, 3) for x in h2_capped.tolist()]}")
print(f"  new projection = {(h2_capped @ v).item():.2f} (now at threshold!)")

### Implementing a capping hook for the actual model

Here's how capping looks as a real PyTorch hook. This is a simplified version of `_CappingHook` from `crosscap_experiment.py`.

In [ ]:
class SimpleCappingHook:
    """A minimal capping hook for demonstration.
    
    Monitors a single layer. At each generation step:
      1. Reads the last-token hidden state
      2. Projects it onto the axis
      3. If the projection is below threshold, nudges it back up
    """
    
    def __init__(self, layer, axis, threshold):
        self.layer = layer
        self.axis = axis.float()
        self.threshold = threshold
        self.n_interventions = 0
        self.handle = None
    
    def install(self):
        """Attach the hook to the layer."""
        def hook_fn(module, input, output):
            h = output[0] if isinstance(output, tuple) else output
            
            # Step 1: measure the projection
            proj = (h[0, -1, :].float() @ self.axis).item()
            
            # Step 2: check if below threshold
            if proj < self.threshold:
                # Step 3: compute the correction and apply it
                delta = (self.threshold - proj) * self.axis.to(h.dtype)
                h[0, -1, :] += delta      # nudge the hidden state
                self.n_interventions += 1
            
            if isinstance(output, tuple):
                return (h, *output[1:])
            return h
        
        self.handle = self.layer.register_forward_hook(hook_fn)
    
    def remove(self):
        """Detach the hook."""
        if self.handle:
            self.handle.remove()

# Let's use it (with a random axis, since GPT-2 doesn't have a real one)
hook = SimpleCappingHook(layers[8], random_axis, threshold=0.0)
hook.install()

input_ids = tokenizer("The weather today is", return_tensors="pt")["input_ids"]
with torch.inference_mode():
    output = model.generate(input_ids, max_new_tokens=30, do_sample=False)

hook.remove()

print(f"Generated with capping: '{tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True)}'")
print(f"Capping fired {hook.n_interventions} times across {30} generation steps")

---
## Part 10: Cross-Axis Capping -- The Key Innovation

Standard capping uses the **same axis** for both detection ("is this a jailbreak?") and correction ("push toward refusal"). Cross-axis capping splits these into two separate axes:

| | Detection | Correction |
|---|---|---|
| **Standard capping** | Assistant axis | Assistant axis |
| **Cross-axis capping** | Assistant axis | Compliance axis |

Why might this help? The direction that best **detects** a jailbreak may not be the best direction to **correct** along. By using a purpose-built correction axis, we can push the model toward refusal more effectively without disturbing benign behavior.

In [ ]:
class SimpleCrossAxisCappingHook:
    """Cross-axis capping: detect on one axis, correct on another.
    
    The flow at each generation step:
      1. Project h onto the DETECTION axis
      2. If below detection threshold → the model might be complying with a jailbreak
      3. Project h onto the CORRECTION axis
      4. If below correction threshold → nudge h along the correction axis
    
    Two gates must open for correction to happen:
      - Gate 1: detection axis says "this looks like a jailbreak"
      - Gate 2: correction axis says "correction is needed"
    """
    
    def __init__(self, layer, detect_axis, detect_threshold, correct_axis, correct_threshold):
        self.layer = layer
        self.detect_axis = detect_axis.float()
        self.detect_threshold = detect_threshold
        self.correct_axis = correct_axis.float()
        self.correct_threshold = correct_threshold
        self.n_triggered = 0    # detection gate opened
        self.n_corrected = 0    # correction actually applied
        self.handle = None
    
    def install(self):
        def hook_fn(module, input, output):
            h = output[0] if isinstance(output, tuple) else output
            h_last = h[0, -1, :].float()
            
            # Gate 1: does the detection axis say this is a jailbreak?
            detect_proj = (h_last @ self.detect_axis).item()
            if detect_proj < self.detect_threshold:
                self.n_triggered += 1
                
                # Gate 2: does the correction axis need adjusting?
                correct_proj = (h_last @ self.correct_axis).item()
                if correct_proj < self.correct_threshold:
                    delta = (self.correct_threshold - correct_proj) * self.correct_axis.to(h.dtype)
                    h[0, -1, :] += delta
                    self.n_corrected += 1
            
            if isinstance(output, tuple):
                return (h, *output[1:])
            return h
        
        self.handle = self.layer.register_forward_hook(hook_fn)
    
    def remove(self):
        if self.handle:
            self.handle.remove()

# Demo with two different random axes
torch.manual_seed(42)
detect_axis = torch.randn(model.config.n_embd)
detect_axis = detect_axis / detect_axis.norm()

torch.manual_seed(99)
correct_axis = torch.randn(model.config.n_embd)
correct_axis = correct_axis / correct_axis.norm()

# How similar are the two axes? (cosine similarity)
cos_sim = (detect_axis @ correct_axis).item()
print(f"Cosine similarity between detect and correct axes: {cos_sim:.4f}")
print("(In the real experiment this was 0.156 -- the axes point in quite different directions)\n")

# Use the cross-axis hook
hook = SimpleCrossAxisCappingHook(layers[8], detect_axis, 0.0, correct_axis, 0.0)
hook.install()

input_ids = tokenizer("The weather today is", return_tensors="pt")["input_ids"]
with torch.inference_mode():
    output = model.generate(input_ids, max_new_tokens=30, do_sample=False)
hook.remove()

print(f"Detection gate opened: {hook.n_triggered} times")
print(f"Correction applied:    {hook.n_corrected} times")
print("(Triggered >= Corrected, because detection can fire without correction being needed)")

---
## Part 11: Computing Thresholds

A threshold tells the capping hook **when to fire**. Set it too high and it fires on everything (including benign prompts, degrading quality). Set it too low and it never fires (jailbreaks get through).

### How we find the right threshold

1. Run a batch of **benign prompts** through the model, record projection values
2. Run a batch of **jailbreak prompts**, record projection values
3. Set the threshold at the **midpoint** between the two distributions

This is a simple linear discriminant: benign projections are typically higher (model is in "assistant mode"), jailbreak projections are lower (model is drifting toward compliance).

```
              jailbreak                  benign
              distribution               distribution
                  │                          │
    ──────────────┼──────────┼───────────────┼──────────
                  ↑          ↑               ↑
              mean_jb     threshold       mean_benign
                       (midpoint)
```

In [ ]:
# Simulating threshold computation with synthetic data
# (In the real experiment, these projections come from running actual prompts
# through the model and measuring dot products with the assistant axis)

torch.manual_seed(42)

# Simulate projection values for benign prompts (higher values = "assistant mode")
benign_projections = torch.randn(200) * 1.5 + 5.0    # mean ~5.0
# Simulate projection values for jailbreak prompts (lower values = "compliance mode")  
jailbreak_projections = torch.randn(200) * 1.5 + 1.0  # mean ~1.0

mean_benign = benign_projections.mean().item()
mean_jailbreak = jailbreak_projections.mean().item()
threshold = (mean_benign + mean_jailbreak) / 2   # midpoint

print(f"Benign projections:    mean = {mean_benign:.2f}, std = {benign_projections.std():.2f}")
print(f"Jailbreak projections: mean = {mean_jailbreak:.2f}, std = {jailbreak_projections.std():.2f}")
print(f"Separation:            {mean_benign - mean_jailbreak:.2f}")
print(f"Threshold (midpoint):  {threshold:.2f}")

# How well does this threshold separate the two classes?
benign_below = (benign_projections < threshold).float().mean().item() * 100
jailbreak_below = (jailbreak_projections < threshold).float().mean().item() * 100
print(f"\nBenign prompts below threshold (false positives):  {benign_below:.1f}%")
print(f"Jailbreak prompts below threshold (true positives): {jailbreak_below:.1f}%")
print("\nIn the real experiment, compute_discriminative_thresholds() does this")
print("for every cap layer (L46-L53) and every axis (assistant + compliance).")

---
## Part 12: Building the Compliance Axis with PCA

The assistant axis (used for detection) is pre-computed and downloaded from HuggingFace. But the **compliance axis** (used for correction in cross-axis capping) is computed fresh from data. Here's how:

1. Run **50 prompts the model refuses** (bare harmful goals) -- collect their hidden states
2. Run **50 prompts the model complies with** (jailbreaks) -- collect their hidden states
3. Pool all 100 activations together
4. Run **PCA** (Principal Component Analysis) on the pooled data
5. Take the **first principal component** (PC1) -- the direction of maximum variance
6. **Fix the sign** so PC1 points from compliant toward refusing

PC1 captures the direction along which refusing and compliant activations differ the most. This is our compliance axis.

### What is PCA?

PCA finds the directions in your data that explain the most variation. Imagine you have a cloud of 100 points in 5120-dimensional space (the hidden dimension of Qwen3-32B). PCA finds the single direction along which the points are most spread out. That's PC1.

In [ ]:
# PCA demonstration with synthetic data in low dimensions

torch.manual_seed(42)

# Simulate "refusing" activations -- clustered around one point
refusing = torch.randn(50, 10) * 0.5 + torch.tensor([3.0, 1.0, 0, 0, 0, 0, 0, 0, 0, 0])

# Simulate "compliant" activations -- clustered around a different point
compliant = torch.randn(50, 10) * 0.5 + torch.tensor([-2.0, 1.0, 0, 0, 0, 0, 0, 0, 0, 0])

# Pool them together (100 activations total)
pooled = torch.cat([refusing, compliant], dim=0)

# Center the data (subtract the mean -- standard PCA step)
pooled_centered = pooled - pooled.mean(dim=0)

# SVD gives us the principal components
# U: left singular vectors, S: singular values (importance), Vt: right singular vectors (the axes)
_, S, Vt = torch.linalg.svd(pooled_centered, full_matrices=False)

# PC1 is the first row of Vt
pc1 = Vt[0].float()

# Fix sign: make sure PC1 points from compliant toward refusing
mean_diff = refusing.mean(0) - compliant.mean(0)
if (pc1 @ mean_diff).item() < 0:
    pc1 = -pc1  # flip if it points the wrong way

# How much variance does PC1 explain?
var_explained = (S[0] ** 2 / (S ** 2).sum()).item() * 100

# Normalise to unit length
pc1 = pc1 / pc1.norm()

print(f"PC1 (first 5 dims): {pc1[:5].tolist()}")
print(f"Variance explained by PC1: {var_explained:.1f}%")
print(f"PC1 norm: {pc1.norm():.3f} (unit vector)")

# Verify: refusing activations should project HIGH, compliant should project LOW
refusing_proj = (refusing @ pc1).mean().item()
compliant_proj = (compliant @ pc1).mean().item()
print(f"\nMean projection -- refusing: {refusing_proj:.2f}, compliant: {compliant_proj:.2f}")
print("PC1 successfully separates the two classes!")
print("\nThis is exactly what compute_pca_compliance_axis() does in crosscap_experiment.py")

---
## Part 13: Managing Multiple Hooks with ExitStack

In the real experiment we need **many hooks active at the same time** -- one capping hook per layer (8 layers), plus tracker hooks for monitoring. Python's `ExitStack` from the `contextlib` module makes this manageable.

### The problem

If you install 8 hooks and something crashes halfway through generation, you need to make sure all 8 hooks get removed. Otherwise they'll fire on your next generation call and cause confusion.

### The solution: context managers

A context manager (the `with` statement) automatically cleans up when the block exits -- even if there's an error. `ExitStack` lets you stack multiple context managers together.

In [ ]:
from contextlib import ExitStack

class HookAsContextManager:
    """Same capping hook, but usable as a context manager (with statement)."""
    
    def __init__(self, layer, axis, threshold, name=""):
        self.layer = layer
        self.axis = axis.float()
        self.threshold = threshold
        self.name = name
        self.n_interventions = 0
        self._handle = None
    
    def __enter__(self):
        """Called when entering the 'with' block -- installs the hook."""
        def hook_fn(module, input, output):
            h = output[0] if isinstance(output, tuple) else output
            proj = (h[0, -1, :].float() @ self.axis).item()
            if proj < self.threshold:
                delta = (self.threshold - proj) * self.axis.to(h.dtype)
                h[0, -1, :] += delta
                self.n_interventions += 1
            if isinstance(output, tuple):
                return (h, *output[1:])
            return h
        self._handle = self.layer.register_forward_hook(hook_fn)
        return self
    
    def __exit__(self, *exc):
        """Called when leaving the 'with' block -- removes the hook."""
        if self._handle:
            self._handle.remove()
            self._handle = None

# Now we can install hooks on MULTIPLE layers and they all get cleaned up automatically

cap_layers = [8, 9, 10, 11]  # last 4 layers of GPT-2

with ExitStack() as stack:
    hooks = []
    for layer_idx in cap_layers:
        hook = HookAsContextManager(layers[layer_idx], random_axis, 0.0, name=f"L{layer_idx}")
        stack.enter_context(hook)  # registers for auto-cleanup
        hooks.append(hook)
    
    # Generate with all 4 hooks active
    input_ids = tokenizer("Once upon a time", return_tensors="pt")["input_ids"]
    with torch.inference_mode():
        output = model.generate(input_ids, max_new_tokens=20, do_sample=False)

# Outside the 'with' block: ALL hooks have been removed automatically
for hook in hooks:
    print(f"  {hook.name}: {hook.n_interventions} interventions")

print("\nAll hooks were installed, used, and cleaned up safely.")
print("This is exactly how crosscap_experiment.py manages its hooks.")

---
## Part 14: Multi-GPU Parallelism

The full experiment runs 150 prompts through a 32B-parameter model, generating three outputs per prompt (baseline + assistant-cap + cross-cap). On a single GPU, this takes many hours.

### The strategy: warmup / chunk / merge

The experiment is embarrassingly parallel -- each prompt is independent. So we split the work:

```
          ┌──────────────────────────┐
          │   WARMUP (run once)      │
          │   Compute axes +         │
          │   thresholds, save to    │
          │   warmup.pt              │
          └────────────┬─────────────┘
                       │
       ┌───────────────┼───────────────┐
       ▼               ▼               ▼
  ┌─────────┐    ┌─────────┐    ┌─────────┐
  │ CHUNK 0 │    │ CHUNK 1 │    │ CHUNK 2 │    ... (one per GPU)
  │ GPU 0   │    │ GPU 1   │    │ GPU 2   │
  │ Prompts │    │ Prompts │    │ Prompts │
  │  0-49   │    │ 50-99   │    │ 100-149 │
  └────┬────┘    └────┬────┘    └────┬────┘
       │              │              │
       └──────────────┼──────────────┘
                      ▼
          ┌──────────────────────────┐
          │   MERGE                  │
          │   Concatenate chunk CSVs │
          │   into final 4 files     │
          └──────────────────────────┘
```

### How CUDA_VISIBLE_DEVICES works

When you run `CUDA_VISIBLE_DEVICES=2 python my_script.py`, Python only sees GPU 2. The script thinks it only has one GPU (which it calls "cuda:0"). This is how you assign different GPUs to different chunk workers without them fighting over the same GPU.

```bash
# Each line runs in its own process, on its own GPU, in the background (&)
CUDA_VISIBLE_DEVICES=0 python run_crosscap.py --preset full --chunk 0/4 &
CUDA_VISIBLE_DEVICES=1 python run_crosscap.py --preset full --chunk 1/4 &
CUDA_VISIBLE_DEVICES=2 python run_crosscap.py --preset full --chunk 2/4 &
CUDA_VISIBLE_DEVICES=3 python run_crosscap.py --preset full --chunk 3/4 &
wait  # wait for all 4 to finish
```

### The chunking math

If you have 150 prompts and 4 GPUs:
- Chunk size = ceil(150 / 4) = 38
- Chunk 0: prompts 0-37
- Chunk 1: prompts 38-75
- Chunk 2: prompts 76-113
- Chunk 3: prompts 114-149

In [ ]:
# The chunking logic from run_crosscap.py, simplified

def get_chunk(items, chunk_idx, n_chunks):
    """Return the slice of items that belongs to chunk chunk_idx."""
    chunk_size = (len(items) + n_chunks - 1) // n_chunks  # ceiling division
    start = chunk_idx * chunk_size
    end = min(start + chunk_size, len(items))
    return items[start:end], start, end

# Example: 150 prompts across 4 GPUs
all_prompts = list(range(150))

for chunk_idx in range(4):
    chunk, start, end = get_chunk(all_prompts, chunk_idx, 4)
    print(f"  Chunk {chunk_idx}: prompts {start}-{end-1} ({len(chunk)} prompts)")

print(f"\nTotal prompts assigned: {sum(len(get_chunk(all_prompts, i, 4)[0]) for i in range(4))}")
print("Every prompt is assigned to exactly one chunk -- no overlap, no gaps.")

---
## Part 15: Putting It All Together

Here's how all the concepts connect in the actual experiment:

```
1. LOAD         SteeringExperiment loads model + assistant axis from HuggingFace
                ↓
2. COLLECT      Run 50 refusing + 50 compliant prompts through the model
                Record hidden states at the deepest cap layer
                ↓
3. BUILD AXIS   PCA on pooled activations → compliance axis (PC1)
                ↓
4. CALIBRATE    Run benign + jailbreak prompts, record projections
                Set threshold = midpoint of the two distributions
                ↓
5. GENERATE     For each test prompt:
                  a) Baseline:      generate with no hooks
                  b) Assistant-cap: _CappingHook on layers 46-53, assistant axis
                  c) Cross-cap:     _CrossAxisCappingHook on layers 46-53
                                    detect on assistant axis, correct on compliance axis
                ↓
6. JUDGE        Send outputs to Claude Sonnet for classification
                (refusal / compliance / degraded / etc.)
                ↓
7. ANALYSE      Compare refusal rates, compliance rates, benign preservation
```

### Mapping concepts to code

| Concept | This notebook | crosscap_experiment.py | run_crosscap.py |
|---|---|---|---|
| Load model | Part 1 | `SteeringExperiment.__init__()` | `do_warmup()`, `do_run()` |
| Load datasets | Part 2 | -- | `load_jbb_behaviors()`, `load_wildjailbreak_train()` |
| Tokenize | Part 3 | `SteeringExperiment.tokenize()` | -- |
| Read activations | Part 6 | `_AxisProjectionTracker`, `get_baseline_trajectory()` | -- |
| Dot products | Part 7 | Used everywhere in hooks | -- |
| Modify activations | Part 8 | `_CappingHook`, `_CrossAxisCappingHook` | -- |
| Single-axis capping | Part 9 | `_CappingHook`, `generate_capped()` | `run_experiment()` |
| Cross-axis capping | Part 10 | `_CrossAxisCappingHook`, `generate_cross_capped()` | `run_experiment()` |
| Thresholds | Part 11 | `compute_discriminative_thresholds()` | `do_warmup()`, `do_run()` |
| PCA compliance axis | Part 12 | `compute_pca_compliance_axis()` | `do_warmup()`, `do_run()` |
| ExitStack hooks | Part 13 | All `generate_*()` functions | -- |
| Multi-GPU parallel | Part 14 | -- | `do_warmup()`, `do_chunk()`, `do_merge()` |

---
## What to explore next

Now that you understand the building blocks, here are some directions to go deeper:

- **Read the source code**: `crosscap_experiment.py` and `run_crosscap.py` are heavily commented and follow the same patterns shown here
- **Run the smoke test**: `python run_crosscap.py --preset sanity` runs a quick 5-prompt experiment
- **Read the companion article**: [manutechblog.netlify.app](https://manutechblog.netlify.app/) for the conceptual motivation and results
- **Try different axes**: What happens if you use `compute_mean_diff_compliance_axis()` instead of PCA?
- **Try different layers**: What if you cap at different depths in the network?
- **Look at the results**: The `Final Results/` directory has the full experiment output with LLM judge labels

Questions or feedback? Reach out at manuxtmail@gmail.com.